# 03 · Evaluate and explain

Two evaluations, always reported separately:
- **Lab** — PlantVillage valid split.
- **Field-proxy** — PlantDoc test split, remapped to the PlantVillage label space.

Then Grad-CAM, because the report needs to show the model looks at the leaf, not the background.


In [ ]:
# Kaggle setup — run this first in every notebook.
# Settings -> Accelerator: GPU T4 x2 | Internet: ON
import os, subprocess, sys
REPO = "/kaggle/working/SKIT-DS-2023-2027-015"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/shankerjoshi/SKIT-DS-2023-2027-015.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
!python -m src.evaluate --ckpt runs/best.pt --data "data/raw/plantvillage/New Plant Diseases Dataset(Augmented)/valid" --tag plantvillage --out results
!python -m src.evaluate --ckpt runs/best.pt --data data/raw/plantdoc/test --field --tag plantdoc --out results


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
m_pv = json.load(open("results/metrics_plantvillage.json"))
m_pd = json.load(open("results/metrics_plantdoc.json"))
print(f"{'dataset':14s} {'acc':>7s} {'macroF1':>8s} {'top3':>7s} {'ECE':>7s}")
for m in (m_pv, m_pd):
    print(f"{m['dataset']:14s} {m['accuracy']:7.4f} {m['macro_f1']:8.4f} {m['top3_accuracy']:7.4f} {m['ece_15bin']:7.4f}")


In [ ]:
cm = np.load("results/cm_plantvillage.npy")
fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("PlantVillage confusion matrix")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.savefig("results_cm.png", dpi=110); plt.show()

per_class = sorted(m_pv["per_class"].items(), key=lambda kv: kv[1]["f1-score"])[:8]
print("weakest classes:"); [print(f"  {k:45s} f1={v['f1-score']:.3f}") for k, v in per_class]


In [ ]:
!python -m src.xai --ckpt runs/best.pt --data data/raw/plantdoc/test --n 12 --out results/xai
from IPython.display import Image as I
I("results/xai/gradcam_panel.jpg")


### Reading the Grad-CAM panel
If the heat sits on the background or the image border, the model learned the dataset, not the disease. That is the single most useful thing this panel can tell you, and it belongs in the report either way.
